# B3 · Multiclass classification: handwritten digits

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 45 minutes · **core path ★**

## Objective

Solve a 10-class problem with a logistic: choose `C` with `LogisticRegressionCV`, compare the multinomial approach with **one-vs-rest**, summarize the
performance with the right averages (macro, per class) and identify the main confusions.

## Dataset

`load_digits`: 1797 8×8 images (64 pixels) of digits from 0 to 9, stratified 70/30 split: `X_tr, X_te, y_tr, y_te`.

## Educational purpose

Bring to the multiclass case everything you saw in the binary one: what changes in `coef_`, in `predict_proba`, in the metrics and how a 10×10 confusion matrix is read.

## Task

1. Build `cv_model`: pipeline `StandardScaler` + `LogisticRegressionCV(Cs=np.logspace(-4, 1, 6), cv=5, scoring="neg_log_loss", max_iter=3000)`. Train it and save `chosen_C` (float).
2. With that `C` train `multinomial` (`LogisticRegression`) and `ovr` (`OneVsRestClassifier(LogisticRegression(...))`), both in a pipeline with a scaler. Save in `table` (DataFrame, index `multinomial` and `ovr`) the columns `accuracy`, `macro_f1` and `log_loss` on the test set.
3. From the multinomial model get `per_class_recall` (array of 10) and `hardest_class` (the digit with the lowest recall).
4. Save `main_confusion`: the pair `(true, predicted)` with the largest number of errors (off the diagonal) in the confusion matrix of the test set.
5. Save `coef_shape` (shape of `coef_` of the multinomial) and `prob_sum` (sum of the predicted probabilities for the first test image).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import confusion_matrix, f1_score, log_loss, recall_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X_tr, X_te, y_tr, y_te = train_test_split(digits.data, digits.target, test_size=0.3, stratify=digits.target, random_state=0)

## Your work

In [ ]:
cv_model = ...
chosen_C = None

In [ ]:
multinomial = ...
ovr = ...
table = pd.DataFrame(index=["multinomial", "ovr"], columns=["accuracy", "macro_f1", "log_loss"], dtype=float)
table

In [ ]:
per_class_recall = None
hardest_class = None
main_confusion = None
coef_shape = None
prob_sum = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`chosen_C = float(cv_model[-1].C_[0])`. `cm = confusion_matrix(y_te, pred); np.fill_diagonal(cm, 0); np.unravel_index(cm.argmax(), cm.shape)`. `recall_score(y_te, pred, average=None)`.'):
    check(isinstance(chosen_C, float) and any(np.isclose(chosen_C, c) for c in np.logspace(-4, 1, 6)), "C chosen among the grid values", "chosen_C = float(cv_model[-1].C_[0])")
    check(table.notna().all().all() and table.loc["multinomial", "accuracy"] > 0.95, "multinomial accuracy above 0.95", "use the same C for both models")
    check(table.loc["multinomial", "log_loss"] < table.loc["ovr", "log_loss"], "the multinomial has a better log-loss than one-vs-rest", "the multinomial probabilities sum to 1 by construction, the ovr ones are normalized afterwards")
    check(close(table.loc["multinomial", "macro_f1"], f1_score(y_te, multinomial.predict(X_te), average="macro")), "correct macro F1", "average='macro'")
    check(len(per_class_recall) == 10 and hardest_class == int(np.argmin(per_class_recall)) and hardest_class == 8, "the hardest digit is the 8", "recall_score(..., average=None)")
    check(main_confusion == (8, 1), "the most frequent error is 8 mistaken for 1", "confusion matrix with the diagonal zeroed, then argmax")
    check(tuple(coef_shape) == (10, 64) and close(prob_sum, 1.0), "coef_ has one row per class and the probabilities sum to 1", "shape (10, 64)")
done()

---
The commented solution is in `solutions/B_with_sklearn/B3_multiclass_digits_and_choice_of_c.ipynb`. Open it after you pass the check, or after you have really tried.